# Model Mission End-User Audit

## tl;dr

Model Mission scores **7.3/10** as a student-facing configurable ML builder. All **8 generated variants executed their coded scope**, **6 were end-to-end workflows**, and the live interface had **no page overflow or panel overlap across five measured viewport widths**. The largest trust gaps are the untrained neural-network exports, YOLO controls that do not behave as the interface implies, inconsistent final-test semantics, and single-file project packaging.

## Context & Methods

This is a diagnostic companion to the reader-facing audit report. It summarizes the live UI inventory, generated scripts, public-dataset profiles, runtime logs, responsive measurements, and the repository's focused automated test suite.

### Key Assumptions

- A successful smoke run establishes integration and executability, not production model quality.
- Deep-learning runs used one epoch and deliberately small public datasets.
- “End-to-end” means the generated script loaded data, trained, evaluated, and produced an artifact or inference result.
- Scores are weighted product judgments grounded in the recorded evidence; they are not statistical estimates.

In [1]:
from pathlib import Path
import json
import pandas as pd

evidence_path = Path("2026-07-28-model-mission-audit-evidence.json")
evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
evidence["audit_date"], evidence["route"]

('2026-07-28', '/tools/ai-script-generator/')

## Data

### 1. Review tested workflows and datasets

In [2]:
task_results = pd.DataFrame(evidence["task_results"])
task_results[[
    "task", "variant", "dataset", "dataset_size", "runtime_status",
    "scope", "duration_seconds", "primary_metric", "metric_value"
]]

,task,variant,dataset,dataset_size,runtime_status,scope,duration_seconds,primary_metric,metric_value
0,Classification,scikit-learn + SMOTE,UCI Adult,"32,561 rows",pass,end-to-end,4.156,test accuracy,0.789800
1,Regression,scikit-learn random forest,UCI Auto MPG,398 rows,pass,end-to-end,1.422,test R²,0.923000
2,Sensor classification,PyTorch LSTM,UCI HAR,"7,352 windows",pass,end-to-end,9.140,test accuracy,0.323663
3,Image classification,TensorFlow MobileNetV3Small,TensorFlow Flowers mini subset,100 images / 5 classes,pass,end-to-end,22.422,validation accuracy,0.000000
4,Object detection,Ultralytics YOLOv8n,COCO8,4 train / 4 validation images,pass,end-to-end,20.328,explicit validation mAP50-95,0.291958
5,Instance segmentation,Ultralytics YOLOv8n-seg,COCO8-Seg,4 train / 4 validation images,pass,end-to-end,20.141,explicit mask mAP50-95,0.203333
6,Neural network,Keras tabular MLP,Digits-shaped configuration,64 inputs / 10 classes,pass,architecture-only,3.859,trainable parameters,6570.000000
7,Neural network,PyTorch tabular MLP,Digits-shaped configuration,64 inputs / 10 classes,pass,architecture-only,2.406,training epochs executed,0.000000


In [3]:
dataset_inventory = pd.DataFrame(evidence["dataset_inventory"])
dataset_inventory

,workflow,dataset,grain,size,quality_check,license_or_terms,status
0,Classification,UCI Adult,one person per row,"32,561 rows / 15 columns","4,262 missing values deliberately exercised",UCI dataset terms,executed
1,Regression,UCI Auto MPG,one car per row,398 rows / 9 columns,6 missing horsepower values deliberately exerc...,UCI dataset terms,executed
2,Sensor classification,UCI HAR,128-sample accelerometer window,"7,352 windows / 6 activities",Class coverage and window shape checked,UCI dataset terms,executed
3,Image classification,TensorFlow Flowers,one labeled image,100-image deterministic mini subset / 5 classes,20 images per class,Dataset-specific source terms,executed
4,Object detection,COCO8,one annotated image,4 train / 4 validation images,Ultralytics parser found 17 validation instances,AGPL-3.0 code; COCO image/annotation terms,executed
5,Instance segmentation,COCO8-Seg,one polygon-annotated image,4 train / 4 validation images,Ultralytics parser found 17 validation instances,AGPL-3.0 code; COCO image/annotation terms,executed
6,Neural architecture,Digits-shaped metadata,64 numeric inputs / one digit label,"1,797 public rows available but not loaded by ...",Used only to verify compatible input/output sh...,scikit-learn bundled dataset terms,not connected


## Results

### 2. Reconcile execution coverage

In [4]:
execution_summary = pd.Series({
    "generated_variants": len(task_results),
    "runtime_passes": int((task_results["runtime_status"] == "pass").sum()),
    "end_to_end_variants": int((task_results["scope"] == "end-to-end").sum()),
    "architecture_only_variants": int((task_results["scope"] == "architecture-only").sum()),
    "automated_tests_passed": evidence["automated_tests_passed"],
})
execution_summary

generated_variants             8
runtime_passes                 8
end_to_end_variants            6
architecture_only_variants     2
automated_tests_passed        57
dtype: int64

### 3. Check learning-level differentiation

In [5]:
level_comparison = pd.DataFrame(evidence["level_comparison"])
level_comparison

,task_id,guided_equals_customize,customize_equals_advanced,guided_control_count,customize_control_count,advanced_control_count
0,classification,False,True,5,15,15
1,regression,True,True,14,14,14
2,sensor-classification,False,True,9,22,22
3,image-classification,False,True,6,16,16
4,object-detection,False,True,6,19,19
5,instance-segmentation,False,True,6,19,19
6,neural-network,True,True,16,16,16


### 4. Check responsive containment

In [6]:
responsive = pd.DataFrame(evidence["responsive_results"])
responsive

,viewport,width_px,horizontal_overflow_px,config_visible,code_visible,config_code_overlap_area,offscreen_workflow_steps,small_touch_targets
0,phone-360,360,0,True,False,0,6,3
1,phone-390,390,0,True,False,0,6,3
2,tablet-768,768,0,True,False,0,3,3
3,reported-1128,1128,0,True,True,0,0,5
4,desktop-1440,1440,0,True,True,0,0,5


In [7]:
assert responsive["horizontal_overflow_px"].eq(0).all()
assert responsive["config_code_overlap_area"].eq(0).all()
{
    "measured_widths": responsive["width_px"].tolist(),
    "page_overflow_failures": int((responsive["horizontal_overflow_px"] > 0).sum()),
    "panel_overlap_failures": int((responsive["config_code_overlap_area"] > 0).sum()),
}

{'measured_widths': [360, 390, 768, 1128, 1440],
 'page_overflow_failures': 0,
 'panel_overlap_failures': 0}

### 5. Recalculate the weighted product score

In [8]:
scorecard = pd.DataFrame(evidence["scorecard"])
scorecard["weighted_points"] = scorecard["score"] * scorecard["weight"]
calculated_score = round(scorecard["weighted_points"].sum(), 1)
assert calculated_score == evidence["overall_score"]
scorecard[["dimension", "score", "weight", "weighted_points", "evidence"]]

,dimension,score,weight,weighted_points,evidence
0,Generated-code executability,8.5,0.16,1.360,8/8 generated variants completed their coded s...
1,Configuration depth,7.4,0.14,1.036,"Strong task-specific controls, but several use..."
2,Workflow breadth,7.0,0.12,0.840,"Seven student-facing tasks cover classical, vi..."
3,Learning design,7.4,0.12,0.888,Linear workflow and field help are strong; lev...
4,Data handling,7.0,0.12,0.840,Classical inspection/cleaning is useful; group...
5,Evaluation correctness,6.2,0.12,0.744,"Metrics run, but YOLO validation thresholding ..."
6,Reproducibility,7.0,0.08,0.560,Seeds and exports are present; install command...
7,Responsive usability,8.5,0.08,0.680,No page overflow or panel overlap across five ...
8,Project packaging,5.5,0.06,0.330,"The handoff is still a .py file, not a complet..."


### 6. Rank actionable findings

In [9]:
issues = pd.DataFrame(evidence["issues"]).sort_values(
    ["severity_order", "affected_tasks"], ascending=[False, False]
)
issues[[
    "priority", "finding", "affected_tasks", "confidence", "risk", "recommendation"
]]

,priority,finding,affected_tasks,confidence,risk,recommendation
3,High,Final-test semantics are inconsistent,5,High,Students can compare numbers that have differe...,Use one split contract with task-specific grou...
1,High,YOLO learning-rate control is ignored,2,High,The UI suggests a configuration changed when t...,Expose optimizer choice or set a concrete opti...
2,High,YOLO validation uses inference confidence,2,High,Reported metrics are not comparable to standar...,Separate inference confidence from validation ...
0,High,Neural builder saves untrained models,1,High,Students may mistake an architecture artifact ...,"Connect dataset, split, preprocessing, trainin..."
4,Medium,Advanced is identical to Customize,7,High,Progressive disclosure promises depth that is ...,Merge the levels until Advanced adds genuinely...
9,Medium,Download is not a full project,7,High,"Beginners still assemble environment, folders,...","Download a zip with script, config, requiremen..."
5,Medium,Dependency constraints are dropped,6,High,Future installs can generate incompatibilities...,Render pinned compatible ranges and include re...
6,Medium,Classical output filenames collide,2,High,One project run can overwrite another without ...,"Expose output directory and run name, and gene..."
7,Medium,SMOTE follows dense one-hot encoding,1,Medium,Synthetic fractional one-hot categories can be...,"Offer class weights first, SMOTENC for mixed d..."
8,Medium,Sensor splitting lacks subject/time safeguards,1,High,Related windows can cross partitions and infla...,"Add group-aware, chronological, gap-aware, and..."


## Takeaways

- **The implementation is materially stronger than the previous audit.** Classification, regression, explicit data inspection, three-way splitting, SMOTE, a linear guided builder, and neural layer design now exist.
- **Execution is strong but configuration integrity is not yet perfect.** All generated variants ran, yet YOLO ignored the configured learning rate and validation reused the inference confidence threshold.
- **The neural builder is not an end-to-end ML workflow yet.** It creates and saves an untrained architecture.
- **The responsive layout is fixed at the measured widths.** The remaining narrow-screen issue is discoverability of the horizontally scrollable workflow rail, not overlap.
- **A realistic no-code contribution is about 62% of a typical supported project today.** Reaching roughly 90% requires complete project packaging, verified data contracts, full neural training, tuning/comparison, and honest task-specific evaluation—not an LLM.